<center>
<a href="http://www.insa-toulouse.fr/" ><img src="http://www.math.univ-toulouse.fr/~besse/Wikistat/Images/logo-insa.jpg" style="float:left; max-width: 120px; display: inline" alt="INSA"/></a> 

</center>

# Projet Machine Learning — 4MA 2025-2026
## Prédiction du Risque de Maladie Cardiovasculaire
### INSA Toulouse
---

**Jeu de données** : *Cardiovascular Disease Risk Prediction Dataset* : 15 000 patients   
**Langage** : Python   
**Objectifs** :
- Partie 1 : Prédiction de `Heart_Disease_Risk` 
- Partie 2 : Prédiction de `Cholesterol_LDL` 

> Ce notebook vient compléter l'étude réalisée en R. Les conclusions sont comparables, > mais Python offre une exécution plus rapide, notamment pour la validation croisée.

### 0. Importation des bibliothèques et chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

randomseed = 111 

In [ ]:
# Suppression des warnings pour une sortie plus propre
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("healthcare_synthetic_data.csv")
print(df.shape)
df.head()


In [ ]:
# Vérification des types de données et des valeurs manquantes
print("Types des variables :")
print(df.dtypes)
print("\nValeurs manquantes :")
print(df.isnull().sum())

# Prétraitement des données
df = df.drop(columns=["Patient_ID"])
for col in ["Gender", "Smoking_Status", "Family_History", "Heart_Disease_Risk"]:
    df[col] = pd.Categorical(df[col], ordered=False)
df["Alcohol_Consumption"] = pd.Categorical(
    df["Alcohol_Consumption"], categories=[0, 1, 2], ordered=True)
df["Physical_Activity_Level"] = pd.Categorical(
    df["Physical_Activity_Level"], categories=[0, 1, 2, 3], ordered=True)
print(df.dtypes)
df.describe()


### 1. Prédiction du risque d’accident cardiaque

In [ ]:
X = df.drop(columns=["Heart_Disease_Risk"])
y = df["Heart_Disease_Risk"]

In [ ]:
# Division en ensembles d'apprentissage et de test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=randomseed)

# Vérification des tailles des ensembles
print("Taille apprentissage :", X_train.shape[0])
print("Taille test :", X_test.shape[0])

# Normalisation
scaler = StandardScaler()
scaler.fit(X_train)
Xr_train = scaler.transform(X_train)
Xr_test = scaler.transform(X_test)



Cette étape est essentielle car elle permet d'estimer les performances réelles du modèle sur des données qu'il n'a jamais vues durant l'apprentissage. Sans cette séparation, le modèle pourrait simplement mémoriser les données d'entraînement et afficher de bonnes performances sans pour autant être capable de généraliser à de nouveaux patients.On retient 20% des données pour le test, soit 3 000 patients.

#### 1.1 Modèle Linéaire Généralisé

##### 1.1.1 Sans pénalisation et sans séléction des variables 

Commentaire pour expliquer que dans ce cas on va pas faire la selection des variables ...

In [ ]:
ModelLin = LogisticRegression(random_state=randomseed, max_iter=1000)
ModelLin.fit(Xr_train, y_train)
score = ModelLin.score(Xr_test, y_test)
print("Score du modèle de régression logistique :", score)
y_pred = ModelLin.predict(Xr_test)
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred))

In [ ]:
# Visualisation des oefficients
coef = pd.Series(ModelLin.coef_[0], index=X.columns)
print(f"Variables éliminées : {sum(coef == 0)}")
coef.sort_values().plot(kind='barh', figsize=(8,6), color='teal')
plt.title("Coefficients - Régression Logistique sans pénalisation")
plt.axvline(0, color='powderblue', linestyle='--')
plt.tight_layout()
plt.show()

##### 1.1.2 Pénalisation LASSO L1 

In [ ]:
from sklearn.model_selection import GridSearchCV

param = [{"C": [0.001,0.01, 0.1, 1, 10, 100]}]
logit = GridSearchCV(LogisticRegression(penalty="l1",solver="liblinear"), param_grid=param, cv=10,n_jobs=-1)
logit0pt=logit.fit(Xr_train, y_train)
logit0pt.best_params_["C"]
print("Meilleur score :",logit0pt.best_score_)
print("Meilleur paramètre C :", logit0pt.best_params_)

yChap = logit0pt.predict(Xr_test)
print("Score du modèle de régression logistique optimisé :", accuracy_score(y_test, yChap))


In [ ]:
# Visualisation des coefficients
coef_lasso = pd.Series(logit0pt.best_estimator_.coef_[0], index=X.columns)
print(f"Variables éliminées : {sum(coef_lasso == 0)}")
coef_lasso.sort_values().plot(kind='barh', figsize=(8,6), color='salmon')
plt.title("Coefficients - Régression Logistique Lasso (L1)")
plt.axvline(0, color='red', linestyle='--')
plt.tight_layout()
plt.show()

Avec le C optimal trouvé par validation croisée, le Lasso ne met aucun coefficient à zéro. Les résultats avec pénalisation Lasso (C optimal) sont proches de celles sans pénalisation. Pour un peu pousser l'effet de Lasso sur notre modèle on teste différente valeur de C et voir les variables qui on étés supprimées au bout d'un moment

In [ ]:
# Effet de C sur la sélection de variables 
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    logit_test = LogisticRegression(penalty="l1", solver="liblinear",
                                     C=C, max_iter=1000)
    logit_test.fit(Xr_train, y_train)
    n_zero = sum(logit_test.coef_[0] == 0)
    acc = accuracy_score(y_test, logit_test.predict(Xr_test))
    print(f"C={C:6} | Variables éliminées : {n_zero:2} | Accuracy : {acc:.4f}")